In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

paths = [
    "/content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_train_data.csv",
    "/content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_train_embeddings.csv",
    "/content/drive/My Drive/Cybersecurity/malware_project/train_embeddings_374Features.csv",
    "/content/drive/My Drive/Cybersecurity/malware_project/train_374Features.csv",
    "/content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_data.csv",
    "/content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_embeddings.csv"
]

all_ok = True
for p in paths:
    if os.path.exists(p):
        print(f"✔ OK: {p}")
    else:
        print(f"❌ ERROR: File not found → {p}")
        all_ok = False

if all_ok:
    print("\nAll files are successfully accessible.")
else:
    print("\nSome files are missing — fix paths first!")


✔ OK: /content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_train_data.csv
✔ OK: /content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_train_embeddings.csv
✔ OK: /content/drive/My Drive/Cybersecurity/malware_project/train_embeddings_374Features.csv
✔ OK: /content/drive/My Drive/Cybersecurity/malware_project/train_374Features.csv
✔ OK: /content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_data.csv
✔ OK: /content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_embeddings.csv

All files are successfully accessible.


In [ ]:
# Load CSVs and Embeddings

import pandas as pd
import numpy as np

# DDoS
ddos_df   = pd.read_csv("/content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_data.csv")
ddos_emb  = pd.read_csv("/content/drive/My Drive/Cybersecurity/Cybersecurity_DDoS/ddos_train_embeddings.csv").values

# Zero-Day
zd_df     = pd.read_csv("/content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_train_data.csv")
zd_emb    = pd.read_csv("/content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_train_embeddings.csv").values

# Malware
mal_df    = pd.read_csv("/content/drive/My Drive/Cybersecurity/malware_project/train_374Features.csv")
mal_emb   = pd.read_csv("/content/drive/My Drive/Cybersecurity/malware_project/train_embeddings_374Features.csv").values

print("Data Loaded Successfully")

Data Loaded Successfully


In [ ]:
# Convert BENIGN → 0, apply label remapping to 23 classes

# -------------------
# DDoS label mapping
# -------------------
ddos_labels = ddos_df["Label"].copy()

# DDoS BENIGN = 0 already → keeps 0
# DDoS attacks 1–4 → keep as 1–4

# -------------------
# Zero-day label mapping
# -------------------
zd_labels = zd_df["Label"].copy()

# BENIGN (0) → stays 0
# Others: shift +4 to start from 5
zd_labels = zd_labels.apply(lambda x: 0 if x == 0 else x + 4)

# -------------------
# Malware label mapping
# -------------------
mal_labels = mal_df["Class"].copy()

# BENIGN (4) → map to 0
mal_labels = mal_labels.replace(4, 0)

# Malware attacks 0–3 → shift +19
mal_labels = mal_labels.apply(lambda x: 0 if x == 0 else x + 19)

# Final class index now: 0–22
print("Labels remapped successfully")

Labels remapped successfully


In [ ]:
# Zero-padding for fusion (Each embedding is 128)

dim = 128

# For DDoS samples
ddos_pad_zd   = np.zeros((ddos_emb.shape[0], dim))
ddos_pad_mal  = np.zeros((ddos_emb.shape[0], dim))
ddos_fused    = np.concatenate([ddos_emb, ddos_pad_zd, ddos_pad_mal], axis=1)

# For Zero-Day samples
zd_pad_ddos   = np.zeros((zd_emb.shape[0], dim))
zd_pad_mal    = np.zeros((zd_emb.shape[0], dim))
zd_fused      = np.concatenate([zd_pad_ddos, zd_emb, zd_pad_mal], axis=1)

# For Malware samples
mal_pad_ddos  = np.zeros((mal_emb.shape[0], dim))
mal_pad_zd    = np.zeros((mal_emb.shape[0], dim))
mal_fused     = np.concatenate([mal_pad_ddos, mal_pad_zd, mal_emb], axis=1)

print("Zero-padding applied. Embeddings fused")

Zero-padding applied. Embeddings fused


In [ ]:
# Combine all TRAIN samples into one dataset

X_train_final = np.vstack([ddos_fused, zd_fused, mal_fused])
y_train_final = np.concatenate([ddos_labels, zd_labels, mal_labels])

print("Final train shapes:")
print(X_train_final.shape)
print(y_train_final.shape)

Final train shapes:
(586532, 384)
(586534,)


In [ ]:
print("X_train_final shape:", X_train_final.shape)
print("y_train_final length:", len(y_train_final))

X_train_final shape: (586532, 384)
y_train_final length: 586534


In [ ]:
print("Before trimming →", len(X_train_final), len(y_train_final))

# Fix mismatch
min_len = min(len(X_train_final), len(y_train_final))
X_train_final = X_train_final[:min_len]
y_train_final = y_train_final[:min_len]

print("After trimming →", len(X_train_final), len(y_train_final))

Before trimming → 586532 586534
After trimming → 586532 586532


In [ ]:
# Save final concatenated TRAIN embeddings & labels

output_dir = "/content/drive/My Drive/Cybersecurity/Embedding_Concatenate"

df_out = pd.DataFrame(X_train_final)
df_out["Label"] = y_train_final

df_out.to_csv(f"{output_dir}/Concatenated_Train_embeddings.csv", index=False)

print(f"Saved successfully → {output_dir}/Concatenated_Train_embeddings.csv")

Saved successfully → /content/drive/My Drive/Cybersecurity/Embedding_Concatenate/Concatenated_Train_embeddings.csv
